# IIT - Sujets Avances : Partitionnement, Repertoires Cause-Effet et Reseaux Elargis

## Navigation

Ce notebook est le **deuxieme volet** de la serie IIT. Il approfondit les concepts introduits dans [IIT-1-IntroToPyPhi](IIT-1-IntroToPyPhi.ipynb).

| Notebook | Contenu | Duree |
|----------|---------|-------|
| [1. IIT-1-IntroToPyPhi](IIT-1-IntroToPyPhi.ipynb) | Reseaux, Phi, CES, macro-subsystemes | 60-90 min |
| **2. AdvancedTopics** (ce notebook) | Partitions MIP, repertoires, MICE, reseaux elargis | 60-90 min |

**Prerequis** : avoir complete le notebook 1 (IIT-1-IntroToPyPhi).

---

## Plan de ce notebook

1. **Rappels et configuration** - Reprise des concepts cles du notebook 1
2. **Partitionnement et MIP** - Comment PyPhi cherche la coupe minimale d'information
3. **Repertoires cause-effet** - Distributions de probabilites sur les causes et effets possibles
4. **MICE et concepts** - Mechanismes maximalement irreductibles
5. **Big Phi vs Small Phi** - Difference entre integration systeme et integration mecanisme
6. **Reseaux elargis** - Systemes a 4+ noeuds et interpretation
7. **Performance et coarse-graining** - Strategies pour traiter de grands systemes
8. **Vers IIT 4.0** - Apercu des evolutions recentes de la theorie

<a id="sec1"></a>
## 1. Rappels et configuration

Dans le notebook precedent, nous avons vu :

- **Reseau (Network)** : un ensemble de noeuds binaires connectes, decrit par une TPM
- **Sous-systeme (Subsystem)** : un sous-ensemble de noeuds dans un etat donne
- **Phi (big Phi, $\Phi$)** : mesure de l'integration irreductible du systeme
- **CES (Cause-Effect Structure)** : l'ensemble des concepts du systeme
- **Concept** : un mecanisme avec son cause et son effet maximalement irreductibles

Reprenons avec notre reseau XOR et configurons PyPhi.

In [1]:
import warnings
warnings.filterwarnings("ignore", message=".*pkg_resources is deprecated.*", category=UserWarning)

import os
os.environ['PYPHI_WELCOME_OFF'] = 'yes'

import pyphi
import numpy as np

pyphi.config.VALIDATE_SUBSYSTEM_STATES = False

print("PyPhi version:", pyphi.__version__)
print("Configuration prete.")

PyPhi version: 1.2.0
Configuration prete.


In [2]:
# Creation du reseau XOR (3 noeuds) - identique au notebook 1
network = pyphi.examples.xor_network()
state = (0, 0, 0)
subsystem = pyphi.Subsystem(network, state)

print("Reseau XOR : 3 noeuds (A, B, C)")
print("Etat initial :", state)
print("Noeuds du sous-systeme :", subsystem.node_indices)
print("Connectivite (CM) :")
print(network.cm)

Reseau XOR : 3 noeuds (A, B, C)
Etat initial : (0, 0, 0)
Noeuds du sous-systeme : (0, 1, 2)
Connectivite (CM) :
[[0 1 1]
 [1 0 1]
 [1 1 0]]


<a id="sec2"></a>
## 2. Partitionnement et MIP (Minimum Information Partition)

Le coeur du calcul de $\Phi$ repose sur la recherche de la **partition minimale d'information (MIP)**. La MIP est la facon de couper le systeme en deux qui **detruit le moins d'information**. Si cette coupe minimale detruit beaucoup d'information, c'est que le systeme est fortement integre.

### 2.1. Types de partitions

PyPhi supporte plusieurs types de partitions :
- **Bipartition** : couper le systeme en 2 parties
- **Tripartition** : couper en 3 parties
- **K-partition** : couper en K parties

Pour chaque partition, on mesure l'information detruite par la coupe. La MIP est celle qui minimise cette perte.

In [3]:
# Calculer l'analyse d'irreductibilite du systeme (SIA)
sia = pyphi.compute.sia(subsystem)

print("=== System Irreducibility Analysis (SIA) ===")
print(f"Big Phi: {sia.phi}")
print(f"Coupe (MIP): {sia.cut}")
print(f"Partie from: {sia.cut.from_nodes}")
print(f"Partie to: {sia.cut.to_nodes}")

Computing concepts:   0%|          | 0/7 [00:00<?, ?it/s]

Computing concepts:  14%|█▍        | 1/7 [00:02<00:14,  2.36s/it]

Evaluating Φ cuts:   0%|          | 0/6 [00:00<?, ?it/s]

Evaluating Φ cuts:  17%|█▋        | 1/6 [00:02<00:12,  2.47s/it]

=== System Irreducibility Analysis (SIA) ===
Big Phi: 1.874999
Coupe (MIP): Cut [B] ━━/ /━━➤ [A, C]
Partie from: (1,)
Partie to: (0, 2)


### 2.2. Interpretation de la MIP

La MIP nous dit **ou le systeme est le plus faiblement integre**. Dans notre reseau XOR :

- Si la coupe separe les noeuds {A, B} de {C}, cela signifie que le lien A,B -> C est le plus faible
- Plus $\Phi$ est eleve, plus le systeme est irreductiblement integre
- Un $\Phi = 0$ signifie que le systeme est completement decomposable (pas d'integration)

### 2.3. Complexite du partitionnement

Le nombre de partitions possibles croit exponentiellement avec la taille du systeme (nombre de Bell). C'est pourquoi le calcul exact de $\Phi$ est **intractable** pour de grands reseaux.

In [4]:
# Enumerer toutes les bipartitions possibles du sous-systeme
from pyphi.partition import bipartition

nodes = subsystem.node_indices
all_biparts = list(bipartition(nodes))
print(f"Nombre de bipartitions pour {len(nodes)} noeuds : {len(all_biparts)}")
print("\nBipartitions possibles :")
for i, bp in enumerate(all_biparts):
    print(f"  {i+1}. {bp[0]} | {bp[1]}")

# Nombre de Bell pour les 6 premiers entiers
bell = [1, 1, 2, 5, 15, 52, 203]
print(f"\nNombres de Bell (nombre total de partitions):")
for n in range(len(bell)):
    print(f"  {n} noeuds -> {bell[n]} partitions")

Nombre de bipartitions pour 3 noeuds : 4

Bipartitions possibles :
  1. () | (0, 1, 2)
  2. (0,) | (1, 2)
  3. (1,) | (0, 2)
  4. (0, 1) | (2,)

Nombres de Bell (nombre total de partitions):
  0 noeuds -> 1 partitions
  1 noeuds -> 1 partitions
  2 noeuds -> 2 partitions
  3 noeuds -> 5 partitions
  4 noeuds -> 15 partitions
  5 noeuds -> 52 partitions
  6 noeuds -> 203 partitions


<a id="sec3"></a>
## 3. Repertoires cause-effet

Les **repertoires cause et effet** sont les distributions de probabilite qui decrivent comment un mecanisme contraint les etats passes (cause) et futurs (effet) d'un purview.

### 3.1. Repertoire cause

Le repertoire cause repond a la question : etant donne le mecanisme dans son etat actuel, quelles sont les probabilites des etats anterieurs possibles du purview ?

In [5]:
# Calculer le repertoire cause pour le mecanisme {A, B} et le purview {A}
mechanism = (0, 1)
purview = (0,)

cause_rep = subsystem.cause_repertoire(mechanism, purview)
print(f"Repertoire cause pour mecanisme {mechanism} -> purview {purview}:")
print(f"  Shape: {cause_rep.shape}")
print(f"  Valeurs: {cause_rep.flatten()}")
print(f"  Somme: {cause_rep.sum():.4f} (doit etre 1.0)")

Repertoire cause pour mecanisme (0, 1) -> purview (0,):
  Shape: (2, 1, 1)
  Valeurs: [0.5 0.5]
  Somme: 1.0000 (doit etre 1.0)


In [6]:
# Calculer le repertoire effet pour le meme mecanisme
effect_rep = subsystem.effect_repertoire(mechanism, purview)
print(f"Repertoire effet pour mecanisme {mechanism} -> purview {purview}:")
print(f"  Shape: {effect_rep.shape}")
print(f"  Valeurs: {effect_rep.flatten()}")
print(f"  Somme: {effect_rep.sum():.4f} (doit etre 1.0)")

print("\nComparaison cause vs effet:")
for i, label in enumerate(['A=0', 'A=1']):
    c_val = cause_rep.flatten()[i]
    e_val = effect_rep.flatten()[i]
    print(f"  {label}: cause={c_val:.3f}, effet={e_val:.3f}")

Repertoire effet pour mecanisme (0, 1) -> purview (0,):
  Shape: (2, 1, 1)
  Valeurs: [0.5 0.5]
  Somme: 1.0000 (doit etre 1.0)

Comparaison cause vs effet:
  A=0: cause=0.500, effet=0.500
  A=1: cause=0.500, effet=0.500


### Interpretation : un mecanisme qui ne specifie aucune information

Les deux repertoires sont **uniformes** (`[0.5, 0.5]`) : le mecanisme {A, B} dans son etat actuel ne contraint **ni** le passe **ni** le futur du purview {A}.

- En **cause**, savoir que {A, B} = (0, 1) ne modifie pas la probabilite des etats anterieurs de A : les deux restent equiprobables.
- En **effet**, ce meme mecanisme ne biaise pas davantage l'etat futur de A.

Autrement dit, ce couple (mecanisme, purview) **ne specifie aucune information** : son repertoire contraint est identique au repertoire non-perturbe (uniforme). C'est exactement ce que la section suivante va quantifier — la distance EMD entre le repertoire contraint et le repertoire non-perturbe sera **nulle**, confirmant l'absence d'information specifiee.

> A l'inverse, un purview informatif produirait une distribution **non uniforme** (par exemple `[0.75, 0.25]`), revelant que le mecanisme penche vers un etat particulier du purview. C'est ce contraste qui fonde la notion de *small phi* ($\varphi$) etudiee plus loin.

### 3.2. Repertoire non-perturbe (unconstrained)

Le repertoire **non-perturbe** est la distribution uniforme - ce qu'on attendrait sans aucune contrainte. La difference entre le repertoire contraint et le repertoire non-perturbe mesure **l'information specifiee** par le mecanisme.

In [7]:
# Repertoire non-perturbe (uniforme)
unconstrained = subsystem.unconstrained_cause_repertoire(purview)
print(f"Repertoire non-perturbe pour purview {purview}:")
print(f"  Valeurs: {unconstrained.flatten()}")

from pyphi.distance import hamming_emd
distance = hamming_emd(cause_rep, unconstrained)
print(f"\nDistance EMD (cause contrainte vs non-perturbe): {distance:.4f}")
print("Cette valeur mesure l'information cause specifiee par le mecanisme.")

Repertoire non-perturbe pour purview (0,):
  Valeurs: [0.5 0.5]

Distance EMD (cause contrainte vs non-perturbe): 0.0000
Cette valeur mesure l'information cause specifiee par le mecanisme.


### Exercice 1 : Explorer les repertoires d'un mecanisme different

Calculez les repertoires cause et effet pour le mecanisme `{A, C}` (noeuds 0 et 2) avec le purview `{B, C}` (noeuds 1 et 2).

**Objectifs** :
1. Calculer le repertoire cause et le repertoire effet
2. Verifier que chaque repertoire somme a 1.0
3. Identifier quel etat du purview est le plus probable en cause et en effet

**Indices** :
- Utilisez `subsystem.cause_repertoire(mechanism, purview)` et `subsystem.effect_repertoire(mechanism, purview)`
- Les noeuds sont indices par des tuples d'entiers : `(0, 2)` pour {A, C}
- Pour identifier l'etat le plus probable, utilisez `np.argmax(rep.flatten())`

In [8]:
# Exercice 1 : Repertoires cause-effet pour le mecanisme {A, C} et purview {B, C}
# TODO etudiant : remplacez les valeurs None par votre code

mechanism_ex1 = None  # TODO : definir le mecanisme {A, C}
purview_ex1 = None    # TODO : definir le purview {B, C}

if mechanism_ex1 is not None and purview_ex1 is not None:
    cause_rep_ex1 = subsystem.cause_repertoire(mechanism_ex1, purview_ex1)
    effect_rep_ex1 = subsystem.effect_repertoire(mechanism_ex1, purview_ex1)
    
    print(f"Repertoire cause: {cause_rep_ex1.flatten()}")
    print(f"Somme cause: {cause_rep_ex1.sum():.4f}")
    print(f"Etat le plus probable (cause): {np.argmax(cause_rep_ex1.flatten())}")
    print(f"\nRepertoire effet: {effect_rep_ex1.flatten()}")
    print(f"Somme effet: {effect_rep_ex1.sum():.4f}")
    print(f"Etat le plus probable (effet): {np.argmax(effect_rep_ex1.flatten())}")
else:
    print("Exercice a completer : definissez mechanism_ex1 et purview_ex1")

Exercice a completer : definissez mechanism_ex1 et purview_ex1


<a id="sec4"></a>
## 4. MICE et Concepts

Un **MICE** (Maximally Irreducible Cause or Effect) est un couple (mecanisme, purview) qui maximise l'information irreductible. Un **concept** est forme d'un MICE cause et d'un MICE effet pour le meme mecanisme.

### 4.1. Trouver le MICE pour un mecanisme

In [9]:
# Trouver le MICE (cause et effet) pour le mecanisme {A, B}
mechanism = (0, 1)

mice_cause = subsystem.find_mice(pyphi.Direction.CAUSE, mechanism)
mice_effect = subsystem.find_mice(pyphi.Direction.EFFECT, mechanism)

print("=== MICE pour le mecanisme {A, B} ===")
print("\nMICE Cause:")
print(f"  Purview: {mice_cause.purview}")
print(f"  Phi (small): {mice_cause.phi}")
print(f"  Direction: {mice_cause.direction}")
print("\nMICE Effet:")
print(f"  Purview: {mice_effect.purview}")
print(f"  Phi (small): {mice_effect.phi}")
print(f"  Direction: {mice_effect.direction}")

=== MICE pour le mecanisme {A, B} ===

MICE Cause:
  Purview: (0, 1, 2)
  Phi (small): 0.5
  Direction: CAUSE

MICE Effet:
  Purview: (2,)
  Phi (small): 0.5
  Direction: EFFECT


In [10]:
# Construire les concepts a partir de la CES
ces = pyphi.compute.ces(subsystem)
print(f"Nombre total de concepts dans la CES: {len(ces)}")
print("\nDetails de chaque concept :")
for i, concept in enumerate(ces):
    print(f"\n  Concept {i+1}:")
    print(f"    Mecanisme: {concept.mechanism}")
    if concept.cause:
        print(f"    Cause purview: {concept.cause.purview}, phi={concept.cause.phi:.4f}")
    if concept.effect:
        print(f"    Effet purview: {concept.effect.purview}, phi={concept.effect.phi:.4f}")
    print(f"    Phi du concept: {concept.phi:.4f}")

Computing concepts:   0%|          | 0/7 [00:00<?, ?it/s]

Nombre total de concepts dans la CES: 3

Details de chaque concept :

  Concept 1:
    Mecanisme: (0, 1)
    Cause purview: (0, 1, 2), phi=0.5000
    Effet purview: (2,), phi=0.5000
    Phi du concept: 0.5000

  Concept 2:
    Mecanisme: (0, 2)
    Cause purview: (0, 1, 2), phi=0.5000
    Effet purview: (1,), phi=0.5000
    Phi du concept: 0.5000

  Concept 3:
    Mecanisme: (1, 2)
    Cause purview: (0, 1, 2), phi=0.5000
    Effet purview: (0,), phi=0.5000
    Phi du concept: 0.5000


### 4.2. Interpretation des MICE

Chaque concept decrit une **unite d'information cause-effet** :
- Le **mecanisme** est l'ensemble de noeuds qui specifie de l'information
- Le **purview cause** est l'ensemble de noeuds dont le mecanisme contraint les etats passes
- Le **purview effet** est l'ensemble de noeuds dont le mecanisme contraint les etats futurs
- Le **small phi** ($\varphi$) mesure la quantite d'information irreductible du concept

Un concept avec $\varphi > 0$ existe dans la CES du systeme. La somme de tous les $\varphi$ des concepts ne donne PAS le big $\Phi$.

<a id="sec5"></a>
## 5. Big Phi vs Small Phi

La distinction entre **big Phi** ($\Phi$) et **small phi** ($\varphi$) est fondamentale :

| | Big Phi ($\Phi$) | Small phi ($\varphi$) |
|---|---|---|
| **Niveau** | Systeme entier | Mecanisme individuel |
| **Mesure** | Integration irreductible du systeme | Information irreductible d'un concept |
| **Question** | Le systeme est-il integre ? | Ce mecanisme specifie-t-il de l'information ? |
| **Calcul** | SIA sur toutes les partitions | EMD entre repertoire contraint et partitionne |

In [11]:
# Comparaison Big Phi vs Small Phi
print("=== Big Phi (niveau systeme) ===")
print(f"  Phi du systeme (SIA): {sia.phi:.4f}")
print(f"  Coupe minimale: {sia.cut}")

print("\n=== Small Phi (niveau concept) ===")
total_small_phi = sum(c.phi for c in ces)
print(f"  Nombre de concepts: {len(ces)}")
print(f"  Somme des small phi: {total_small_phi:.4f}")
for i, c in enumerate(ces):
    print(f"    Concept {i+1}: mecanisme={c.mechanism}, phi={c.phi:.4f}")

print(f"\nBig Phi ({sia.phi:.4f}) != somme des small phi ({total_small_phi:.4f})")
print("Ce sont deux mesures differentes a des niveaux differents.")

=== Big Phi (niveau systeme) ===
  Phi du systeme (SIA): 1.8750
  Coupe minimale: Cut [B] ━━/ /━━➤ [A, C]

=== Small Phi (niveau concept) ===
  Nombre de concepts: 3
  Somme des small phi: 1.5000
    Concept 1: mecanisme=(0, 1), phi=0.5000
    Concept 2: mecanisme=(0, 2), phi=0.5000
    Concept 3: mecanisme=(1, 2), phi=0.5000

Big Phi (1.8750) != somme des small phi (1.5000)
Ce sont deux mesures differentes a des niveaux differents.


### Exercice 2 : Comparer Phi pour deux etats du reseau

Calculez le big Phi et la CES pour le reseau XOR dans l'etat `(1, 1, 1)` et comparez avec l'etat `(0, 0, 0)`.

**Objectifs** :
1. Creer un nouveau sous-systeme pour l'etat `(1, 1, 1)`
2. Calculer le SIA (big Phi) et la CES (concepts)
3. Comparer les resultats avec l'etat `(0, 0, 0)`

**Indices** :
- `pyphi.Subsystem(network, state)` pour creer le sous-systeme
- `pyphi.compute.sia(subsystem)` pour le big Phi
- `pyphi.compute.ces(subsystem)` pour les concepts

In [12]:
# Exercice 2 : Comparer Phi pour deux etats
# TODO etudiant : remplacez les valeurs None par votre code

state_2 = None  # TODO : definir l'etat (1, 1, 1)

if state_2 is not None:
    subsystem_2 = pyphi.Subsystem(network, state_2)
    sia_2 = pyphi.compute.sia(subsystem_2)
    ces_2 = pyphi.compute.ces(subsystem_2)
    
    print(f"Etat {state}: Big Phi = {sia.phi:.4f}, Concepts = {len(ces)}")
    print(f"Etat {state_2}: Big Phi = {sia_2.phi:.4f}, Concepts = {len(ces_2)}")
    print(f"\nDifference Big Phi: {abs(sia.phi - sia_2.phi):.4f}")
else:
    print("Exercice a completer : definissez state_2 = (1, 1, 1)")

Exercice a completer : definissez state_2 = (1, 1, 1)


<a id="sec6"></a>
## 6. Reseaux elargis : systemes a 4+ noeuds

Explorons un systeme plus grand pour observer comment la complexite de la CES evolue.

### 6.1. Reseau XOR cyclique (4 noeuds)

Nous construisons un reseau a 4 noeuds en anneau ou chaque noeud depend de lui-meme et de son voisin de gauche via XOR.

In [13]:
# Reseau 4 noeuds en anneau - chaque noeud depend de lui-meme et de son voisin
tpm_4 = np.zeros((16, 4))
for i in range(16):
    bits = [(i >> j) & 1 for j in range(3, -1, -1)]
    a_next = bits[3] ^ bits[0]  # D XOR A
    b_next = bits[0] ^ bits[1]  # A XOR B
    c_next = bits[1] ^ bits[2]  # B XOR C
    d_next = bits[2] ^ bits[3]  # C XOR D
    tpm_4[i] = [a_next, b_next, c_next, d_next]

cm_4 = np.array([
    [1, 1, 0, 1],
    [1, 1, 1, 0],
    [0, 1, 1, 1],
    [1, 0, 1, 1],
])

labels_4 = ('A', 'B', 'C', 'D')
net_4 = pyphi.Network(tpm_4, cm_4, node_labels=labels_4)

print("Reseau 4 noeuds en anneau (XOR cyclique)")
print("Labels:", labels_4)
print("Connectivite:")
print(cm_4)

Reseau 4 noeuds en anneau (XOR cyclique)
Labels: ('A', 'B', 'C', 'D')
Connectivite:
[[1 1 0 1]
 [1 1 1 0]
 [0 1 1 1]
 [1 0 1 1]]


In [14]:
# Analyser le reseau 4 noeuds
state_4 = (0, 0, 0, 0)
sub_4 = pyphi.Subsystem(net_4, state_4)

print(f"Sous-systeme 4 noeuds, etat {state_4}")
print(f"Noeuds: {sub_4.node_indices}")

ces_4 = pyphi.compute.ces(sub_4)
print(f"\nNombre de concepts: {len(ces_4)}")
for i, c in enumerate(ces_4):
    cause_p = c.cause.purview if c.cause else None
    effect_p = c.effect.purview if c.effect else None
    print(f"  Concept {i+1}: mech={c.mechanism}, cause_p={cause_p}, "
          f"effect_p={effect_p}, phi={c.phi:.4f}")

Sous-systeme 4 noeuds, etat (0, 0, 0, 0)
Noeuds: (0, 1, 2, 3)


Computing concepts:   0%|          | 0/15 [00:00<?, ?it/s]


Nombre de concepts: 0


In [15]:
# Big Phi pour le reseau 4 noeuds
sia_4 = pyphi.compute.sia(sub_4)
print(f"Big Phi (4 noeuds): {sia_4.phi:.4f}")
print(f"Coupe MIP: {sia_4.cut}")
print(f"\nComparaison:")
print(f"  3 noeuds (XOR): Big Phi = {sia.phi:.4f}")
print(f"  4 noeuds (anneau): Big Phi = {sia_4.phi:.4f}")

Computing concepts:   0%|          | 0/15 [00:00<?, ?it/s]

Computing concepts:   7%|▋         | 1/15 [00:02<00:32,  2.29s/it]

Big Phi (4 noeuds): 0.0000
Coupe MIP: NullCut((0, 1, 2, 3))

Comparaison:
  3 noeuds (XOR): Big Phi = 1.8750
  4 noeuds (anneau): Big Phi = 0.0000


### 6.2. Interpreter les resultats

Quelques observations importantes :

1. **Plus de noeuds ne signifie pas plus de Phi** : la valeur de $\Phi$ depend de la structure causale, pas de la taille
2. **Les reseaux feed-forward ont Phi = 0** : pas de boucle causale = pas d'integration
3. **Les reseaux recurrents** (comme notre anneau) peuvent avoir Phi > 0
4. **La symetrie** de la connectivite influence la distribution des concepts

<a id="sec7"></a>
## 7. Performance et coarse-graining

Le calcul exact de $\Phi$ croit de maniere **super-exponentielle** avec la taille du reseau. Pour des reseaux de plus de ~6-8 noeuds, le calcul devient intractable.

### 7.1. Strategies de gestion de la complexite

| Strategie | Principe | Utilite |
|-----------|----------|----------|
| **Coarse-graining** | Grouper les noeuds en macro-noeuds | Reduire la dimensionnalite |
| **Blackboxing** | Ignorer certains noeuds | Se concentrer sur les noeuds pertinents |
| **Parallelisation** | Repartir les calculs | Accelerer sur multi-coeur |
| **Caching** | Stocker les resultats intermediaires | Eviter les recalculs |

In [16]:
# Demonstration du module macro
print("=== Module pyphi.macro ===")
print("Le module macro offre des outils pour le coarse-graining et blackboxing.")
print("\nFonctions disponibles:")
macro_funcs = [f for f in dir(pyphi.macro) if not f.startswith('_')]
for f in macro_funcs[:12]:
    print(f"  - {f}")

=== Module pyphi.macro ===
Le module macro offre des outils pour le coarse-graining et blackboxing.

Fonctions disponibles:
  - Blackbox
  - CoarseGrain
  - ConditionallyDependentError
  - MacroNetwork
  - MacroSubsystem
  - NodeLabels
  - StateUnreachableError
  - Subsystem
  - SystemAttrs
  - all_blackboxes
  - all_coarse_grains
  - all_coarse_grains_for_blackbox


In [17]:
# Mesurer le temps de calcul pour differentes tailles
import time

def time_phi_calc(network, state, label=""):
    """Mesure le temps de calcul de la CES pour un sous-systeme."""
    sub = pyphi.Subsystem(network, state)
    t0 = time.time()
    ces_result = pyphi.compute.ces(sub)
    elapsed = time.time() - t0
    print(f"  {label}: {len(ces_result)} concepts, {elapsed:.2f}s")
    return elapsed

print("Temps de calcul CES par taille de reseau:")
t3 = time_phi_calc(network, (0,0,0), "3 noeuds (XOR)")
t4 = time_phi_calc(net_4, (0,0,0,0), "4 noeuds (anneau)")

if t3 > 0:
    print(f"\nRatio 4noeuds/3noeuds: {t4/t3:.1f}x")
    print("La croissance est super-lineaire.")

Temps de calcul CES par taille de reseau:


Computing concepts:   0%|          | 0/7 [00:00<?, ?it/s]

  3 noeuds (XOR): 3 concepts, 0.02s


Computing concepts:   0%|          | 0/15 [00:00<?, ?it/s]

  4 noeuds (anneau): 0 concepts, 0.06s

Ratio 4noeuds/3noeuds: 3.8x
La croissance est super-lineaire.


### 7.2. Implications pour la recherche en neuroscience

Cette complexite computationnelle a des implications directes :

- Le **cerveau humain** (~86 milliards de neurones) ne peut pas etre analyse avec le calcul exact
- Les **approximations** (coarse-graining, echantillonnage) sont necessaires
- Le **Perturbational Complexity Index (PCI)** est une approximation clinique inspiree de $\Phi$
- Le debat entre **exactitude mathematique** et **applicabilite pratique** est central

<a id="sec8"></a>
## 8. Vers IIT 4.0

### 8.1. D'IIT 3.0 a IIT 4.0

| Aspect | IIT 3.0 (PyPhi) | IIT 4.0 (2024+) |
|--------|-----------------|------------------|
| **Identite** | Le complexe maximise $\Phi$ | Le systeme EST sa structure cause-effet |
| **Existence intrinseque** | Via les concepts | Axiome fondamental |
| **Coupes** | Bipartitions | Coupes directionnelles |
| **Extension** | Potentiel a Phi maximal | Ensemble des systemes qui existent |

In [18]:
# Apercu : le concept-style (vers IIT 4.0)
try:
    cs_sia = pyphi.compute.sia_concept_style(subsystem)
    print("=== Concept-Style SIA (vers IIT 4.0) ===")
    print(f"Phi (concept-style): {cs_sia.phi:.4f}")
    print(f"Phi (classique 3.0): {sia.phi:.4f}")
    diff = abs(cs_sia.phi - sia.phi)
    print(f"Difference: {diff:.4f}")
    print("Le concept-style utilise un schema de partition different.")
except Exception as e:
    print(f"Concept-style non disponible: {e}")

Computing concepts:   0%|          | 0/7 [00:00<?, ?it/s]

Computing concepts:  14%|█▍        | 1/7 [00:02<00:13,  2.17s/it]

Evaluating Φ cuts:   0%|          | 0/31 [00:00<?, ?it/s]

Evaluating Φ cuts:   3%|▎         | 1/31 [00:02<01:06,  2.20s/it]

Evaluating Φ cuts:   0%|          | 0/31 [00:00<?, ?it/s]

Evaluating Φ cuts:   3%|▎         | 1/31 [00:02<01:06,  2.22s/it]

=== Concept-Style SIA (vers IIT 4.0) ===
Phi (concept-style): 0.6250
Phi (classique 3.0): 1.8750
Difference: 1.2500
Le concept-style utilise un schema de partition different.


### Exercice 3 : Explorer un reseau feed-forward

Construisez un reseau feed-forward a 3 noeuds (A -> B -> C) et verifiez que son big Phi est egal a 0.

**Objectifs** :
1. Creer la TPM d'un reseau ou A influence B et B influence C (pas de boucle)
2. Creer le reseau PyPhi avec la connectivite appropriee
3. Calculer le big Phi et verifier qu'il vaut 0

**Indices** :
- La matrice de connectivite est triangulaire : `[[0,1,0], [0,0,1], [0,0,0]]`
- Pour la TPM : B_next = A, C_next = B, A_next = A (ou une constante)
- Utilisez `pyphi.compute.sia(subsystem)` pour calculer big Phi

In [19]:
# Exercice 3 : Reseau feed-forward (Phi attendu = 0)
# TODO etudiant : construisez le reseau et verifiez Phi = 0

# Etape 1 : Definir la TPM (8 etats -> 3 noeuds)
tpm_ff = None  # TODO : construire la TPM du reseau feed-forward

# Etape 2 : Matrice de connectivite
cm_ff = None   # TODO : A->B, B->C (pas de connexion C->A)

if tpm_ff is not None and cm_ff is not None:
    net_ff = pyphi.Network(tpm_ff, cm_ff)
    sub_ff = pyphi.Subsystem(net_ff, (0, 0, 0))
    sia_ff = pyphi.compute.sia(sub_ff)
    print(f"Big Phi du reseau feed-forward: {sia_ff.phi:.4f}")
    print(f"Phi = 0 confirme: {sia_ff.phi == 0.0}")
    print("Un reseau sans boucle causale n'a pas d'integration.")
else:
    print("Exercice a completer : definissez tpm_ff et cm_ff")

Exercice a completer : definissez tpm_ff et cm_ff


### 8.2. Limites et debats

L'IIT est une theorie active et debattue :

- **Lettre ouverte de 2023** : ~120 signataires qualifiant l'IIT de pseudoscience
- **Reponse des partisans** : l'IIT fait des predictions falsifiables (PCI, activations)
- **Programme Templeton** : tests adversariaux IIT vs Global Workspace Theory
- **Enjeu IA** : l'IIT predit que les reseaux feed-forward (comme les LLMs) ont $\Phi = 0$

Ces debats illustrent la tension entre **rigueur mathematique** et **applicabilite empirique**.

---

## Bilan et perspectives

Dans ce deuxieme notebook, nous avons approfondi :

1. **Le partitionnement MIP** - la recherche de la coupe minimale d'information
2. **Les repertoires cause-effet** - les distributions de probabilite au coeur de l'analyse IIT
3. **Les MICE et concepts** - les unites d'information irreductible dans la CES
4. **Big Phi vs Small Phi** - deux niveaux de mesure complementaires
5. **Les reseaux elargis** - comment la complexite croit avec la taille du systeme
6. **Le coarse-graining** - strategies pour gerer l'intractabilite du calcul
7. **IIT 4.0** - les evolutions recentes de la theorie

### Pour aller plus loin

- **Documentation PyPhi** : [pyphi.readthedocs.io](https://pyphi.readthedocs.io/en/stable/)
- **Article fondateur** : Oizumi, Albantakis, Tononi (2014). *From the Phenomenology to the Mechanisms of Consciousness*
- **IIT 4.0** : Albantakis et al. (2023). *Integrated information theory (IIT) 4.0*
- **Series connexes** : [Probas](../Probas/README.md), [GameTheory](../GameTheory/README.md)